# RGATE training on Google Colab

Runs `rgate/train_rgate.py` on a Colab GPU. **Backup only:** the main training machine is the RTX 4070 PC
(`rgate/CLAUDE.md` §6d, `rgate/run_all.sh`). Same script and flags; only paths differ.
Plan: two runs, seed 0, training set = BDD100K train night images only: first `ARM = "baseline"`, then
`ARM = "gate"`, `LAM = 0.05`, `GATE_LEVELS` = the decision of the level check (`rgate/level_check.py`: "p3" or "p3p4").

**Before you start**
1. *Runtime → Change runtime type → GPU* (T4 on free Colab).
2. Put these in Google Drive under `MyDrive/rgate/` (see `rgate/CLAUDE.md`, section "Google Colab"):
   - `ultralytics-yolo26.zip` — your repo without `.git`, `dataset/`, `runs/` (or set `REPO_GIT_URL` instead)
   - `data/<name>.zip` — one zip per dataset, each with a single top folder `<name>/` holding `data.yaml`, `images/`, `labels/`

**Every dataset must be in exactly the format of `dataset/sampled_*`** (training set and every eval set):
`data.yaml` with the same 10 class names in the same order, `images/*.jpg`, `labels/*.txt` in YOLO format
(`class cx cy w h`, normalised 0–1). Cell 6 checks every label file and stops on the first problem.
For thesis numbers the training set must come from BDD100K **train** and the eval sets from BDD100K **val**.

**Where results go:** `MyDrive/rgate/runs/<run name>/weights/` — saved to Drive every epoch, so a disconnect loses at most one epoch.
Every new best is kept as `best_ep<NNN>_map50-95-<x>_map50-<y>.pt`, plus `best.pt` and `last.pt`.
A run is finished when its folder has `done.json`.

Run cells 1–6 in every new session. Then run **7 (smoke)** once, then **8 (train)**, or **9 (resume)** after a disconnect.
Every command cell stops with an error if the command fails, so a red error always means "it did not work".

## 1. Settings — the only cell you normally edit

In [ ]:
DRIVE_ROOT  = "/content/drive/MyDrive/rgate"           # folder in your Google Drive
REPO_GIT_URL = ""                                      # e.g. "https://github.com/<you>/ultralytics-yolo26.git"; empty = use the zip below
REPO_ZIP     = f"{DRIVE_ROOT}/ultralytics-yolo26.zip"  # used when REPO_GIT_URL is empty
REFRESH_REPO = True    # True = always take a fresh copy of the code (needed after you upload a new zip / push new code)
DATA_ZIPS    = [                                       # zips in Drive; each unzips to /content/data/<name>/
    f"{DRIVE_ROOT}/data/sampled_bdd100k.zip",          # replace with your full datasets (same format as the samples)
    f"{DRIVE_ROOT}/data/sampled_bdd100k_night.zip",
    f"{DRIVE_ROOT}/data/sampled_bdd100k_night_rain.zip",
]
TRAIN_SET  = "sampled_bdd100k"                                    # folder name (zip name) used for training
EVAL_SETS  = ["sampled_bdd100k_night", "sampled_bdd100k_night_rain"]  # scored with best.pt after training
RUNS_DIR   = f"{DRIVE_ROOT}/runs"                      # outputs on Drive (survive disconnects)

# experiment arm and training settings (cell 8)
ARM     = "gate"        # "gate" or "baseline"
LAM     = 0.05          # gate only: 0 = A1, 0.05 = A2
GATE_LEVELS = "p3p4"    # gate only: "p3" or "p3p4" = the decision of the level check (rgate/level_check.py)
NO_PRIOR = False        # gate only: True = A3 control (needs LAM = 0)
WEIGHTS = "yolo26n.pt"  # official name (downloads) or a .pt path, e.g. f"{DRIVE_ROOT}/weights/best_bdd.pt"
EPOCHS, IMGSZ, BATCH, SEED = 50, 640, 32, 0   # BATCH = -1 lets Ultralytics pick from GPU memory
EXTRA   = ""            # extra flags. "--allow-overlap" ONLY for test runs on the samples (they share 1 image);
                        # never for a real run: it hides eval images that leak into training

REPO = "/content/ultralytics-yolo26"
DATA = "/content/data"

# ---- helper, do not edit ----
import codecs, os, signal, subprocess, sys
def sh(cmd, cwd=None):
    """Run a shell command, show its output live, and STOP with an error if it fails ('!' does not stop).
    Stopping the cell (stop button / Ctrl+M I) also stops the command and everything it started."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1", "COLUMNS": "120"}, start_new_session=True)
    dec = codecs.getincrementaldecoder("utf-8")(errors="replace")
    try:
        while chunk := os.read(p.stdout.fileno(), 4096):
            sys.stdout.write(dec.decode(chunk)); sys.stdout.flush()
    except BaseException:  # cell stopped: without this the training would keep running in the background
        os.killpg(p.pid, signal.SIGTERM)
        try:
            p.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid, signal.SIGKILL)
        print("\n[stopped] the command was stopped; the last finished epoch is in last.pt (use cell 9 to resume)")
        raise
    if p.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}. Read the error above.")

## 2. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU, then rerun."
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUNS_DIR, exist_ok=True)
missing = [z for z in ([] if REPO_GIT_URL else [REPO_ZIP]) + DATA_ZIPS if not os.path.isfile(z)]
for z in ([] if REPO_GIT_URL else [REPO_ZIP]) + DATA_ZIPS:
    print(("MISSING " if z in missing else "OK      ") + z)
assert not missing, "Upload the missing files to Drive first."

## 4. Get the repo (patched Ultralytics + `rgate/`)

In [ ]:
import shutil
if os.path.isdir(REPO) and REFRESH_REPO:
    shutil.rmtree(REPO)  # old code copy from earlier in this session; take the current one
if os.path.isdir(REPO):
    print("Repo already here (REFRESH_REPO = False):", REPO)
elif REPO_GIT_URL:
    sh(f"git clone --depth 1 {REPO_GIT_URL} {REPO}")
else:
    sh(f'cp "{REPO_ZIP}" /content/repo.zip && unzip -q -o /content/repo.zip -d /content && rm /content/repo.zip')
assert os.path.isfile(f"{REPO}/rgate/train_rgate.py"), f"rgate/train_rgate.py not found under {REPO}"
n_patch = open(f"{REPO}/ultralytics/nn/tasks.py").read().count("RGATE")
assert n_patch == 5, f"tasks.py has {n_patch} RGATE markers, expected 5: wrong or unpatched repo"
assert "binary_cross_entropy_with_logits" in open(f"{REPO}/ultralytics/nn/modules/rgate.py").read(), \
    "rgate.py is older than the AMP fix (2026-11-14): re-zip and re-upload the repo"
assert "--resume" in open(f"{REPO}/rgate/train_rgate.py").read(), "train_rgate.py has no --resume: re-upload the repo"
print("code OK")

## 5. Install dependencies (Colab's own PyTorch is kept)

In [ ]:
sh(f"pip install -q -e {REPO} -r {REPO}/rgate/requirements.txt")
sh(f"python -c \"import sys; sys.path.insert(0, '{REPO}'); import ultralytics; print('ultralytics', ultralytics.__version__, ultralytics.__file__)\"")

## 6. Copy datasets to the Colab disk and unzip
Training reads images from `/content/data` (fast local disk), never from Drive (very slow for many small files).
Large zips take a few minutes; this runs once per session.

In [ ]:
os.makedirs(DATA, exist_ok=True)
for z in DATA_ZIPS:
    name = os.path.splitext(os.path.basename(z))[0]
    if os.path.isdir(f"{DATA}/{name}"):
        print("already unzipped:", name); continue
    sh(f'cp "{z}" /content/tmp.zip && unzip -q -o /content/tmp.zip -d {DATA} -x "__MACOSX/*" "*.DS_Store" && rm /content/tmp.zip')
    assert os.path.isfile(f"{DATA}/{name}/data.yaml"), f"{z} must unzip to a single folder {name}/ with data.yaml inside"
    n_img = len(os.listdir(f"{DATA}/{name}/images")) if os.path.isdir(f"{DATA}/{name}/images") else "?"
    print(f"{name}: {n_img} images")
EVAL_ARGS = " ".join(f"{DATA}/{e}" for e in EVAL_SETS)
# checks every label file of every dataset, class names/order, the split and eval/train overlap
sh(f"python rgate/train_rgate.py --data {DATA}/{TRAIN_SET} --eval {EVAL_ARGS} --project {RUNS_DIR} --check-only {EXTRA}", cwd=REPO)

## 7. Smoke test on the GPU (1 epoch, ~1 min). Run once per new setup.

In [ ]:
sh(f"python rgate/train_rgate.py --arm gate --lam 0.05 --gate-levels p3p4 --data {DATA}/{TRAIN_SET} --eval {EVAL_ARGS} "
   f"--project {RUNS_DIR} --smoke", cwd=REPO)

## 8. Train (real run)

In [ ]:
ARM_ARGS = "--arm baseline" if ARM == "baseline" else (f"--arm gate --lam {LAM} --gate-levels {GATE_LEVELS}" + (" --no-prior" if NO_PRIOR else ""))
CMD = (f"python rgate/train_rgate.py {ARM_ARGS} --data {DATA}/{TRAIN_SET} --eval {EVAL_ARGS} "
       f"--weights {WEIGHTS} --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --seed {SEED} --project {RUNS_DIR} {EXTRA}")
sh(CMD, cwd=REPO)

## 9. Resume after a disconnect
Run cells 1–6 again (same settings), then this cell. It continues the newest run in `RUNS_DIR` that has a `last.pt`
and no `done.json` (smoke runs are skipped). To pick a run yourself, set `RESUME` to its `last.pt`.
Keep `ARM`, `LAM`, `NO_PRIOR`, `TRAIN_SET` as they were for that run: the script stops if they do not match the checkpoint
or if the train/val split changed. If training had already finished, it only redoes the evaluation.

In [ ]:
import glob
RESUME = ""  # e.g. f"{RUNS_DIR}/gate-p3p4-l0.05_n_bdd_train_night_s0/weights/last.pt"
if not RESUME:
    cands = [p for p in glob.glob(f"{RUNS_DIR}/*/weights/last.pt")
             if not os.path.isfile(os.path.join(os.path.dirname(os.path.dirname(p)), "done.json"))
             and not os.path.dirname(os.path.dirname(p)).endswith("_smoke")]
    if not cands:
        raise SystemExit("Nothing to resume: every run in RUNS_DIR has done.json (or none saved a last.pt yet). "
                         "Start a new run with cell 8.")
    RESUME = max(cands, key=os.path.getmtime)
print("resuming", RESUME)
ARM_ARGS = "--arm baseline" if ARM == "baseline" else (f"--arm gate --lam {LAM} --gate-levels {GATE_LEVELS}" + (" --no-prior" if NO_PRIOR else ""))
sh(f'python rgate/train_rgate.py {ARM_ARGS} --data {DATA}/{TRAIN_SET} --eval {EVAL_ARGS} --project {RUNS_DIR} {EXTRA} '
   f'--resume "{RESUME}"', cwd=REPO)

## 10. See saved weights and scores

In [ ]:
import glob
for run in sorted(glob.glob(f"{RUNS_DIR}/*/weights")):
    d = os.path.dirname(run)
    print("\n" + d + ("   [done]" if os.path.isfile(f"{d}/done.json") else "   [not finished]"))
    for f in sorted(os.listdir(run)):
        print(f"   {f:55s} {os.path.getsize(os.path.join(run, f))/1e6:6.1f} MB")
    s = os.path.join(d, "eval_summary.csv")
    if os.path.isfile(s):
        print(open(s).read())